In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import warnings

import numpy as np
import pandas as p
import matplotlib.pyplot as pltd
from PIL import Image
import yaml

import torch
import ultralytics
from ultralytics import YOLO
from sklearn.model_selection import train_test_split


# À MODIFIER : dossier contenant dossier_1/, dossier_2/, etc.
ROOT = Path("labelled_data").expanduser().resolve()

# Dossiers de sortie, distincts du dossier source.
PREPARED = Path("dataset_thermal").resolve()
RUNS = Path("runs_thermiques").resolve()

PROFILE = "cpu"            # "cpu" ou "gpu"
SEED = 42

SPLIT_MODE = "group"        # "group" ou "image"
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# False : un label absent est une erreur.
# True : un label absent signifie explicitement "aucune personne".
# Un fichier .txt vide est accepté dans les deux cas.
MISSING_LABELS_ARE_NEGATIVES = False

# Taille commune pour comparer les modèles à la résolution visée.
EVAL_IMGSZ = 640
MAX_DET = 300

print("PyTorch :", torch.__version__)
print("Ultralytics :", ultralytics.__version__)
print("CUDA disponible :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

PyTorch : 2.15.0+cpu
Ultralytics : 8.4.218
CUDA disponible : False


In [3]:
from PIL import Image
import numpy as np


def load_gray8(path):
    """Charge une image 8 bits et la convertit en gris sans la redimensionner."""
    with Image.open(path) as im:
        if getattr(im, "n_frames", 1) != 1:
            raise ValueError(
                "Image multi-page : extraire les images au préalable."
            )

        if im.getexif().get(274, 1) != 1:
            raise ValueError(
                "Orientation EXIF : corriger image ET annotations avant import."
            )

        if im.mode not in ("L", "RGB"):
            raise ValueError(
                f"Mode {im.mode} : image 8 bits en mode L ou RGB attendue."
            )

        # Accepte aussi les images contenant du rouge ou d'autres couleurs.
        # Aucun changement de résolution ni des positions des objets.
        a = np.array(im.convert("L"), dtype=np.uint8)

    if a.ndim != 2 or min(a.shape) < 10:
        raise ValueError(
            "Image attendue : dimensions >= 10 pixels."
        )

    return np.ascontiguousarray(a)


from pathlib import Path
import numpy as np


def read_boxes(label, allow_missing=False):
    """Lit les boîtes YOLO ; classe source exigée : 1 = personne."""
    label = Path(label)

    if not label.is_file():
        if allow_missing:
            return np.empty((0, 4), dtype=float)
        raise ValueError(f"Label manquant : {label}")

    rows = []

    for line_no, line in enumerate(
        label.read_text(encoding="utf-8-sig").splitlines(),
        start=1,
    ):
        if not line.strip():
            continue

        try:
            values = np.asarray(
                [float(value) for value in line.split()],
                dtype=float,
            )
        except ValueError as exc:
            raise ValueError(
                f"{label}, ligne {line_no} : valeur non numérique."
            ) from exc

        if values.shape != (5,) or not np.isfinite(values).all():
            raise ValueError(
                f"{label}, ligne {line_no} : "
                "cinq nombres finis attendus."
            )

        cls, x, y, w, h = values

        # Classe 1 uniquement.
        if cls != 1:
            raise ValueError(
                f"{label}, ligne {line_no} : "
                f"classe reçue {cls}, classe attendue 1 (personne)."
            )

        if not (
            0 <= x <= 1
            and 0 <= y <= 1
            and 0 < w <= 1
            and 0 < h <= 1
        ):
            raise ValueError(
                f"{label}, ligne {line_no} : "
                f"coordonnées invalides : x={x}, y={y}, w={w}, h={h}."
            )

        if (
            min(x - w / 2, y - h / 2) < -1e-6
            or max(x + w / 2, y + h / 2) > 1 + 1e-6
        ):
            raise ValueError(
                f"{label}, ligne {line_no} : boîte hors de l'image."
            )

        rows.append([x, y, w, h])

    boxes = np.asarray(rows, dtype=float).reshape(-1, 4)

    if len(np.unique(boxes, axis=0)) != len(boxes):
        raise ValueError(f"{label} : boîtes exactement dupliquées.")

    return boxes

In [4]:
from pathlib import Path
import hashlib
import warnings

import numpy as np
import pandas as pd


def audit_dataset(root):
    root = Path(root).expanduser().resolve()

    if not root.is_dir():
        raise ValueError(f"Dossier racine introuvable : {root}")

    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    rows = []
    errors = []
    excluded = []
    total_images = 0

    def exclude(folder, reason):
        excluded.append({
            "dossier": folder.name,
            "motif": reason,
        })
        print(f"DOSSIER EXCLU — {folder.name} : {reason}")

    folders = sorted(
        path for path in root.iterdir()
        if path.is_dir()
    )

    for folder in folders:
        image_dir = folder / "img"
        label_dir = folder / "labels"

        # 1. Dossier img absent : aucune image à utiliser.
        if not image_dir.is_dir():
            exclude(folder, "dossier img absent")
            continue

        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file()
            and path.suffix.lower() in extensions
        )

        total_images += len(images)

        if not images:
            exclude(folder, "aucune image dans img")
            continue

        # Un fichier nommé "labels" n'est pas un dossier labels.
        if label_dir.exists() and not label_dir.is_dir():
            errors.append(
                f"{folder.name} : {label_dir} existe, "
                "mais ce n'est pas un dossier."
            )
            continue

        labels_present = label_dir.is_dir()

        # 2. Dossier labels absent : conserver les images négatives.
        if not labels_present:
            print(
                f"SANS DOSSIER LABELS — {folder.name} : "
                f"{len(images)} image(s) conservée(s) comme négatives"
            )

        # 3. Dossier labels présent : signaler les anomalies,
        # sans exclure les images pour label vide ou manquant.
        else:
            label_files = sorted(
                path for path in label_dir.iterdir()
                if path.is_file()
                and path.suffix.lower() == ".txt"
            )

            empty_labels = []
            reading_failed = False

            for label_file in label_files:
                try:
                    text = label_file.read_text(encoding="utf-8-sig")
                except (OSError, UnicodeError) as exc:
                    errors.append(f"{label_file} : {exc}")
                    reading_failed = True
                    continue

                if not text.strip():
                    empty_labels.append(label_file.name)

            # Un seul message par dossier pour les labels vides.
            if empty_labels:
                print(
                    f"LABELS VIDES — {folder.name} : "
                    f"{len(empty_labels)} fichier(s) vide(s), "
                    "aucune exclusion pour ce motif"
                )

            missing_labels = [
                image for image in images
                if not (label_dir / f"{image.stem}.txt").is_file()
            ]

            # Un seul message par dossier pour les labels manquants.
            if missing_labels:
                print(
                    f"LABELS MANQUANTS — {folder.name} : "
                    f"{len(missing_labels)} image(s) sans .txt associé, "
                    "conservée(s) comme négatives"
                )

            # Une annotation illisible reste une erreur bloquante.
            if reading_failed:
                continue

        # 4. Lecture des images et des annotations.
        for image in images:
            label = label_dir / f"{image.stem}.txt"

            try:
                a = load_gray8(image)
                h, w = a.shape

                # Label absent ou vide : tableau vide = image négative.
                # Label non vide : conserver les contrôles habituels.
                boxes = read_boxes(label, allow_missing=True)

                sizes = (
                    boxes[:, 2:]
                    * np.array([w, h])
                    * min(640 / w, 640 / h)
                )

                # Comparaison du contenu des images, pas de leurs noms.
                digest = hashlib.sha256(
                    str(a.shape).encode() + a.tobytes()
                ).hexdigest()

                rows.append({
                    "image": str(image),
                    "label": str(label),
                    "group": folder.name,
                    "width": w,
                    "height": h,
                    "n_boxes": len(boxes),
                    "positive": len(boxes) > 0,
                    "min_side_640": (
                        float(sizes.min()) if len(boxes) else np.nan
                    ),
                    "fingerprint": digest,
                })

            except (ValueError, OSError, UnicodeError) as exc:
                errors.append(f"{image} : {exc}")

    # Les images comptées ici sont les fichiers aux extensions acceptées
    # trouvés directement dans les dossiers img.
    images_kept = len(rows)
    images_excluded = total_images - images_kept

    print("\nBILAN" + (" PROVISOIRE — audit en erreur" if errors else ""))
    print(f"Dossiers examinés : {len(folders)}")
    print(f"Dossiers exclus : {len(excluded)}")
    print(f"Images trouvées : {total_images}")
    print(f"Images exclues / non retenues : {images_excluded}")
    print(f"Total d'images gardées : {images_kept}")

    # Ne pas poursuivre avec des données techniquement invalides.
    if errors:
        raise ValueError(
            "Audit interrompu : corriger les erreurs suivantes.\n"
            + "\n".join(errors[:20])
            + f"\nTotal : {len(errors)} erreur(s)."
        )

    if not rows:
        raise ValueError("Aucune image utilisable.")

    df = pd.DataFrame(rows)

    if df.n_boxes.sum() == 0:
        raise ValueError(
            "Aucune personne annotée dans les images conservées : "
            "le jeu ne contient que des images négatives."
        )

    if df.fingerprint.duplicated().any():
        warnings.warn(
            "Images identiques détectées : vérifier les doublons "
            "et leurs annotations. Aucune suppression automatique."
        )

    return df, pd.DataFrame(
        excluded,
        columns=["dossier", "motif"],
    )


df, dossiers_exclus = audit_dataset(ROOT)

print(f"\nPersonnes annotées : {df.n_boxes.sum()}")
print(f"Images positives : {df.positive.sum()}")
print(f"Images traitées comme négatives : {(~df.positive).sum()}")

print("\nRésolutions présentes :")
print(
    df.groupby(["width", "height"])
      .size()
      .sort_values(ascending=False)
      .rename("images")
      .head(20)
)

print(
    "\nPlus petit côté de boîte par image, "
    "après réduction à imgsz=640 :"
)
print(
    df.min_side_640.describe(percentiles=[0.1, 0.5, 0.9])
)

if df.n_boxes.max() > MAX_DET:
    raise ValueError(
        f"Une image contient {df.n_boxes.max()} objets : "
        "augmenter MAX_DET."
    )

ValueError: Dossier racine introuvable : /home/cytech/Deep Learning/ProjetSDIS/labelled_data

In [ ]:
from pathlib import Path
from collections import Counter

root = Path(ROOT).expanduser().resolve()

if not root.is_dir():
    raise ValueError(f"Dossier introuvable : {root}")

counts = Counter()
files_by_class = {0: set(), 1: set()}
examples_zero = []
n_files = 0
n_empty = 0

# Recherche dans tous les sous-dossiers nommés "labels".
for label in sorted(root.rglob("*.txt")):
    if label.parent.name != "labels":
        continue

    try:
        text = label.read_text(encoding="utf-8-sig")
    except (OSError, UnicodeError) as exc:
        print(f"ERREUR DE LECTURE — {label} : {exc}")
        continue

    n_files += 1

    if not text.strip():
        n_empty += 1
        continue

    for line_no, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue

        # Seul le premier nombre représente la classe.
        try:
            cls = float(line.split()[0])
        except ValueError:
            print(f"CLASSE ILLISIBLE — {label}, ligne {line_no} : {line}")
            continue

        if cls in (0, 1):
            cls = int(cls)
            counts[cls] += 1
            files_by_class[cls].add(label)

            # Un seul exemple par fichier.
            if (
                cls == 0
                and len(examples_zero) < 5
                and all(item[0] != label for item in examples_zero)
            ):
                examples_zero.append((label, line_no, line))
        else:
            counts["autres"] += 1

print(f"\nFichiers lus : {n_files}")
print(f"Fichiers vides : {n_empty}")

print(f"\nAnnotations de classe 0 : {counts[0]}")
print(f"Annotations de classe 1 : {counts[1]}")
print(f"Annotations d'autres classes : {counts['autres']}")

print(f"\nFichiers contenant au moins un 0 : {len(files_by_class[0])}")
print(f"Fichiers contenant au moins un 1 : {len(files_by_class[1])}")

print("\nExemples de fichiers contenant la classe 0 :")

if not examples_zero:
    print("Aucun fichier trouvé avec la classe 0.")
else:
    for path, line_no, line in examples_zero:
        print(f"\nFichier : {path}")
        print(f"Ligne {line_no} : {line}")

In [ ]:
def split_dataset(df, mode="group", val_ratio=.15, test_ratio=.15, seed=42):
    if not (
        0 < val_ratio < 1
        and 0 < test_ratio < 1
        and val_ratio + test_ratio < 1
    ):
        raise ValueError("Ratios invalides.")

    df = df.copy()

    if mode == "group":
        units = sorted(df.group.unique())

        if len(units) < 3:
            raise ValueError(
                "Il faut >= 3 séquences indépendantes. Avec deux dossiers, "
                "définir les vrais groupes, ajouter une séquence, ou choisir "
                "mode='image' uniquement si les images sont indépendantes."
            )

        trval, te = train_test_split(
            units, test_size=test_ratio, random_state=seed
        )
        tr, va = train_test_split(
            trval,
            test_size=val_ratio / (1 - test_ratio),
            random_state=seed,
        )

        mapping = {
            group: name
            for name, groups in (("train", tr), ("val", va), ("test", te))
            for group in groups
        }
        df["split"] = df.group.map(mapping)

    elif mode == "image":
        def split_ids(ids, ratio):
            y = df.loc[ids, "positive"]
            return train_test_split(
                ids,
                test_size=ratio,
                random_state=seed,
                stratify=y if y.nunique() == 2 else None,
            )

        trval, te = split_ids(df.index, test_ratio)
        tr, va = split_ids(trval, val_ratio / (1 - test_ratio))

        df["split"] = ""
        for name, ids in (("train", tr), ("val", va), ("test", te)):
            df.loc[ids, "split"] = name

    else:
        raise ValueError("mode doit être 'group' ou 'image'.")

    # Un doublon exact ne doit jamais traverser les ensembles.
    if (df.groupby("fingerprint")["split"].nunique() > 1).any():
        raise ValueError(
            "Des images identiques traversent les ensembles : "
            "regrouper ou supprimer ces doublons."
        )

    stats = df.groupby("split").agg(
        images=("image", "size"),
        boxes=("n_boxes", "sum"),
        positive_ratio=("positive", "mean"),
        groups=("group", "nunique"),
        min_side_640=("min_side_640", "median"),
    )

    if set(stats.index) != {"train", "val", "test"} or (stats.boxes == 0).any():
        raise ValueError(
            "Chaque ensemble doit contenir des images et des personnes annotées."
        )

    if stats.positive_ratio.max() - stats.positive_ratio.min() > .20:
        warnings.warn(
            "Écart > 20 points entre les proportions d'images avec personnes : "
            "examiner le découpage."
        )

    if stats.images.min() < 30:
        warnings.warn(
            "Au moins un ensemble contient < 30 images : "
            "métriques potentiellement instables."
        )

    stats["image_fraction"] = stats.images / len(df)
    return df, stats


df, split_stats = split_dataset(
    df,
    mode=SPLIT_MODE,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
    seed=SEED,
)

print(split_stats.round(3))

# Optionnel : examiner la répartition détaillée des groupes.
# print(df.groupby(["split", "group"]).size())

In [ ]:
def prepare_dataset(df, target):
    target = Path(target).resolve()

    if target.exists():
        raise FileExistsError(
            f"{target} existe déjà. Réutiliser son data.yaml "
            "ou choisir un autre dossier."
        )

    for split in ("train", "val", "test"):
        (target / "images" / split).mkdir(parents=True)
        (target / "labels" / split).mkdir(parents=True)

    manifest = df.copy()

    for idx, row in df.iterrows():
        # Identifiants uniques : aucun conflit entre les noms des dossiers sources.
        name = f"{idx:08d}"

        image_dst = target / "images" / row.split / f"{name}.png"
        Image.fromarray(load_gray8(row.image)).convert("RGB").save(image_dst)

        label_dst = target / "labels" / row.split / f"{name}.txt"
        source_label = Path(row.label)

        # Annotations inchangées ; suppression du BOM éventuel.
        text = (
            source_label.read_text(encoding="utf-8-sig")
            if source_label.is_file()
            else ""
        )
        label_dst.write_text(text, encoding="utf-8")

        manifest.loc[idx, "prepared_image"] = str(image_dst)

    config = {
        "path": str(target),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "inutilisee",
                  1: "personne"},
        "channels": 3,
    }

    data_yaml = target / "data.yaml"
    data_yaml.write_text(
        yaml.safe_dump(config, sort_keys=False), encoding="utf-8"
    )
    manifest.to_csv(target / "manifest.csv", index=False)

    return data_yaml

PREPARED = Path("dataset_thermal").expanduser().resolve()


# ==========================================================
# CAS 1 — GARDER LE DATASET EXISTANT
# Actif actuellement : aucune image ni annotation n'est modifiée.
# ==========================================================

#DATA_YAML = PREPARED / "data.yaml"

#if not DATA_YAML.is_file():
#    raise FileNotFoundError(
#        f"Configuration introuvable : {DATA_YAML}"
#    )

#print("Dataset existant réutilisé :", PREPARED)
#print("Configuration :", DATA_YAML)


# ==========================================================
# CAS 2 — REGÉNÉRER LE DATASET
# Commenter le CAS 1, puis décommenter ce bloc.
# ATTENTION : supprime entièrement le dossier PREPARED,
# puis le recrée à partir de df et des fichiers sources.
# Exécuter d'abord l'audit et le découpage train/val/test.
# ==========================================================

import shutil

if PREPARED.exists():
    shutil.rmtree(PREPARED)

DATA_YAML = prepare_dataset(df, PREPARED)

print("Dataset régénéré :", PREPARED)
print("Configuration :", DATA_YAML)

In [ ]:
from pathlib import Path
import textwrap
import numpy as np
import matplotlib.pyplot as plt

if df.empty:
    raise ValueError("Le tableau df ne contient aucune image.")

sample = df.sample(n=min(8, len(df)), random_state=SEED)

# Deux images par ligne.
ncols = min(2, len(sample))
nrows = (len(sample) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(16, 6 * nrows),
    squeeze=False,
)
axes = axes.ravel()

for ax, (_, row) in zip(axes, sample.iterrows()):
    image = load_gray8(row.image)
    label_path = Path(row.label)

    # Charger les annotations de CETTE image.
    if label_path.is_file():
        boxes = read_boxes(label_path, allow_missing=False)
        annotation_info = f"{len(boxes)} personne(s)"

    elif not label_path.parent.is_dir():
        boxes = np.empty((0, 4), dtype=float)
        annotation_info = "Image négative — dossier labels absent"

    else:
        boxes = np.empty((0, 4), dtype=float)
        annotation_info = "ATTENTION : fichier label manquant"
        print(f"Label manquant : {label_path}")

    # Afficher l'image dans tous les cas.
    h, w = image.shape
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)

    for cx, cy, bw, bh in boxes:
        x = (cx - bw / 2) * w
        y = (cy - bh / 2) * h

        ax.add_patch(
            plt.Rectangle(
                (x, y),
                bw * w,
                bh * h,
                fill=False,
                edgecolor="lime",
                linewidth=1.5,
            )
        )

    chemin = textwrap.fill(str(row.image), width=75)

    ax.set_title(
        f"{Path(row.image).name}\n"
        f"{row.split} — {annotation_info}\n"
        f"{chemin}",
        fontsize=9,
    )
    ax.axis("off")

# Masquer les emplacements inutilisés.
for ax in axes[len(sample):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tkinter
print("Tk disponible :", tkinter.TkVersion)

In [ ]:
# Code pour annoter images soi-même:
# Dossier qui ont été annoté avec ce code ci-dessous : DJI_20260104183316_0003_3_T, DJI_20260104183842_0006_1_T, DJI_20260104185611_0001_1_T, DJI_20260104185611_0001_2_T, DJI_20260104185611_0001_3_T, DJI_20260104190000_0005_2_T, DJI_20260104190000_0005_4_T
%matplotlib tk

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.widgets import Button
import numpy as np


def annoter_dossier(dossier):
    dossier = Path(dossier).expanduser().resolve()
    image_dir = dossier / "img"
    label_dir = dossier / "labels"

    if not image_dir.is_dir():
        raise ValueError(f"Dossier img introuvable : {image_dir}")

    extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    images = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )

    if not images:
        raise ValueError(f"Aucune image dans : {image_dir}")

    # Évite deux images différentes partageant le même fichier label.
    stems = [p.stem for p in images]
    if len(stems) != len(set(stems)):
        raise ValueError(
            "Plusieurs images ont le même nom sans extension."
        )

    state = {
        "index": 0,
        "points": [],
        "marker": None,
        "image": None,
        "boxes": [],
    }

    fig, ax = plt.subplots(figsize=(12, 9))
    fig.subplots_adjust(bottom=0.23)

    status = fig.text(
        0.05, 0.13, "",
        fontsize=10,
    )

    ax_previous = fig.add_axes([0.12, 0.04, 0.22, 0.055])
    ax_cancel = fig.add_axes([0.39, 0.04, 0.22, 0.055])
    ax_next = fig.add_axes([0.66, 0.04, 0.22, 0.055])

    buttons = [
        Button(ax_previous, "Précédente"),
        Button(ax_cancel, "Annuler clic"),
        Button(ax_next, "Suivante"),
    ]

    def message(text):
        status.set_text(text)
        fig.canvas.draw_idle()

    def draw_box(box, color="lime"):
        cx, cy, bw, bh = box
        h, w = state["image"].shape

        ax.add_patch(
            plt.Rectangle(
                ((cx - bw / 2) * w, (cy - bh / 2) * h),
                bw * w,
                bh * h,
                fill=False,
                edgecolor=color,
                linewidth=1.5,
            )
        )

    def show_image(index):
        path = images[index]
        label = label_dir / f"{path.stem}.txt"

        # Charger avant de changer l'image courante.
        image = load_gray8(path)
        boxes = read_boxes(label, allow_missing=True)

        state.update({
            "index": index,
            "points": [],
            "marker": None,
            "image": image,
            "boxes": [tuple(box) for box in boxes],
        })

        ax.clear()
        ax.imshow(image, cmap="gray", vmin=0, vmax=255)
        ax.set_title(
            f"{index + 1}/{len(images)} — {path.name}\n"
            "Deux clics autour de chaque personne manquante"
        )

        for box in state["boxes"]:
            draw_box(box)

        ax.set_xlim(-0.5, image.shape[1] - 0.5)
        ax.set_ylim(image.shape[0] - 0.5, -0.5)

        message(
            f"{len(boxes)} annotation(s) existante(s)\n"
            f"Label : {label}"
        )

    def cancel_click(event=None):
        state["points"].clear()

        if state["marker"] is not None:
            state["marker"].remove()
            state["marker"] = None

        message("Premier clic annulé. Vous pouvez recommencer.")

    def move(offset):
        index = state["index"] + offset

        if not 0 <= index < len(images):
            message("Début du dossier." if offset < 0 else "Fin du dossier.")
            return

        try:
            show_image(index)
        except (ValueError, OSError) as exc:
            message(f"Impossible de charger l'image : {exc}")

    def clic(event):
        if event.inaxes != ax or event.button != 1:
            return

        if ax.get_navigate_mode() is not None:
            return

        h, w = state["image"].shape
        x = max(0, min(w, event.xdata))
        y = max(0, min(h, event.ydata))

        state["points"].append((x, y))

        if len(state["points"]) == 1:
            state["marker"], = ax.plot(x, y, "r+")
            message("Premier coin placé : cliquez sur le coin opposé.")
            return

        (x1, y1), (x2, y2) = state["points"]
        state["points"].clear()

        if state["marker"] is not None:
            state["marker"].remove()
            state["marker"] = None

        xmin, xmax = sorted((x1, x2))
        ymin, ymax = sorted((y1, y2))

        if xmax <= xmin or ymax <= ymin:
            message("Boîte de taille nulle : recommencez.")
            return

        box = (
            (xmin + xmax) / (2 * w),
            (ymin + ymax) / (2 * h),
            (xmax - xmin) / w,
            (ymax - ymin) / h,
        )

        # Arrondis identiques à ceux enregistrés dans le fichier.
        box = tuple(round(value, 6) for value in box)

        if box[2] <= 0 or box[3] <= 0:
            message("Boîte trop petite : recommencez.")
            return

        if any(
            np.allclose(box, existing, atol=1e-6, rtol=0)
            for existing in state["boxes"]
        ):
            message("Cette boîte existe déjà : aucun ajout.")
            return

        path = images[state["index"]]
        label = label_dir / f"{path.stem}.txt"
        line = "1 " + " ".join(f"{value:.6f}" for value in box)

        try:
            label_dir.mkdir(parents=True, exist_ok=True)

            # Ajouter une ligne sans effacer les annotations existantes.
            existing_text = (
                label.read_text(encoding="utf-8-sig")
                if label.is_file()
                else ""
            )

            with label.open("a", encoding="utf-8") as file:
                if existing_text and not existing_text.endswith(("\n", "\r")):
                    file.write("\n")
                file.write(line + "\n")

        except (OSError, UnicodeError) as exc:
            message(f"ÉCHEC : annotation non enregistrée — {exc}")
            return

        state["boxes"].append(box)
        draw_box(box, color="orange")

        message(
            f"ENREGISTRÉ : {line}\n"
            f"{label}"
        )

    buttons[0].on_clicked(lambda event: move(-1))
    buttons[1].on_clicked(cancel_click)
    buttons[2].on_clicked(lambda event: move(1))

    fig.canvas.mpl_connect("button_press_event", clic)

    show_image(0)
    plt.show()

    # Conserver les boutons en mémoire pour qu'ils restent actifs.
    return fig, buttons

# Chemin du dossier contenant img/ et labels/.
annotateur = annoter_dossier(
    "labelled_data/"
)

In [ ]:
PROFILES = {
    "gpu": {
        "model": "yolo11s.pt",
        "device": 0,          # Premier GPU NVIDIA
        "batch": 8,           # Réduire à 4 ou 2 si mémoire GPU insuffisante
        "epochs": 100,
        "amp": True,          # Précision mixte pour réduire la mémoire GPU
    },
    "cpu": {
        "model": "yolo11n.pt",
        "device": "cpu",
        "batch": 2,
        "epochs": 60,
        "amp": False,
    },
}

# Variantes YOLO11 de taille croissante :
# yolo11n.pt : nano, rapide et léger
# yolo11s.pt : small, compromis intéressant sur GPU
# yolo11m.pt / yolo11l.pt / yolo11x.pt : plus coûteux
#
# Les fichiers .pt utilisent des poids préentraînés.
# Ils sont téléchargés au premier lancement.
AUGMENT = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.3,           # Mettre 0.0 pour désactiver la variation de luminosité

    "fliplr": 0.5,           # Probabilité de retournement horizontal
    "flipud": 0.2,
    "degrees": 5,
    "translate": 0.1,
    "scale": 0.5,        # Zoom
    "shear": 0.0,
    "perspective": 0.0,

    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "augmentations": [
    ],
}

TRAIN = {
    "imgsz": 640,

    "patience": 20,          # Arrêt après 20 époques sans amélioration
    "optimizer": "AdamW",    # Choix explicite : lr0 reste effectivement contrôlé
    "lr0": 1e-3,
    "lrf": 0.01,             # LR final relatif au LR initial
    "weight_decay": 5e-4,
    "warmup_epochs": 3.0,
    "cos_lr": True,

    "rect": False,          # Letterboxing sur un canevas carré
    "multi_scale": 0.0,     # Facultatif : essayer 0.15 après la référence initiale

    "workers": 0,           # Simple et robuste sous Jupyter, notamment Windows
    "cache": False,         # Évite de remplir la RAM du laptop

    "seed": SEED,
    "deterministic": True,
    "max_det": MAX_DET,

    "save": True,
    "val": True,
    "plots": True,

    **AUGMENT,
}


def metrics_row(metrics):
    return {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
    }


def train_and_validate(name, overrides=None):
    cfg = {
        **TRAIN,
        **PROFILES[PROFILE],
        **(overrides or {}),
    }
    initial_weights = cfg.pop("model")

    if cfg["device"] != "cpu" and not torch.cuda.is_available():
        raise RuntimeError(
            "GPU CUDA indisponible : vérifier PyTorch/pilote, "
            "ou choisir PROFILE='cpu'."
        )

    if cfg["device"] == "cpu":
        # Point de départ raisonnable ; à ajuster selon le processeur.
        torch.set_num_threads(
            max(1, min(8, (os.cpu_count() or 2) // 2))
        )

    # Un nouveau modèle préentraîné pour chaque expérience.
    detector = YOLO(initial_weights)

    detector.train(
        data=str(DATA_YAML),
        project=str(RUNS),
        name=f"{PROFILE}_{name}",
        exist_ok=False,
        **cfg,
    )

    run_dir = Path(detector.trainer.save_dir)
    best = run_dir / "weights" / "best.pt"

    if not best.is_file():
        raise FileNotFoundError(f"Checkpoint absent : {best}")

    environment = {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "ultralytics": ultralytics.__version__,
    }
    (run_dir / "environment.json").write_text(
        json.dumps(environment, indent=2),
        encoding="utf-8",
    )

    # Toutes les expériences sont comparées à la MÊME taille d'évaluation.
    # Le jeu de test n'est jamais utilisé ici.
    metrics = YOLO(str(best)).val(
        data=str(DATA_YAML),
        split="val",
        imgsz=EVAL_IMGSZ,
        device=cfg["device"],
        batch=cfg["batch"],
        workers=0,
        conf=0.001,
        iou=0.6,             # Seuil NMS, distinct des seuils IoU de la mAP
        rect=False,
        max_det=MAX_DET,
        plots=True,
        project=str(run_dir),
        name="validation",
        exist_ok=True,
    )

    return {
        "name": name,
        "model": initial_weights,
        "train_imgsz": cfg["imgsz"],
        "weights": str(best),
        **metrics_row(metrics),
    }

In [ ]:
from pathlib import Path

PROFILE = "cpu"

experiments = [
    (
        "nom_experiment",
        {
            "epochs": 60,       # Maximum ; arrêt possible avant
            "imgsz": 640,
            # "freeze": 10,
            "fraction": 1.0,
            "batch": 8,
            "rect": True,

            "val": True,        # Validation à chaque époque
            "patience": 10,     # Early stopping
            "save": True,       # Sauvegarde best.pt et last.pt
            "plots": True,
        },
    ),
]

resultats = []

for name, overrides in experiments:
    # Cette fonction entraîne, récupère best.pt,
    # puis le valide à la taille EVAL_IMGSZ.
    resultat = train_and_validate(name, overrides)
    resultats.append(resultat)

    print("\nMeilleur checkpoint de cette expérience :")
    print(resultat["weights"])

# Si plusieurs expériences : sélectionner celle ayant
# la meilleure mAP50–95 lors de la validation finale.
selection = max(resultats, key=lambda r: r["map50_95"])

# Compatibilité avec tes cellules d'inférence.
BEST = Path(selection["weights"])

print("\nModèle sélectionné pour l'inférence :", BEST)
print("mAP50 :", selection["map50"])
print("mAP50–95 :", selection["map50_95"])

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

for i, bloc in enumerate(model.model.model):
    print(i, bloc.__class__.__name__)

print("Nombre de blocs principaux :", len(model.model.model))

In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo

audit_runs_dir = Path("runs_thermiques").resolve()
audit_runs_info = []

for audit_run in audit_runs_dir.iterdir():
    if not audit_run.is_dir():
        continue

    audit_best = audit_run / "weights" / "best.pt"
    audit_last = audit_run / "weights" / "last.pt"

    audit_present = [
        fichier
        for fichier in (audit_best, audit_last)
        if fichier.is_file()
    ]

    if audit_present:
        audit_runs_info.append((
            max(f.stat().st_mtime for f in audit_present),
            audit_run,
            audit_best,
            audit_last,
        ))

# Classement selon la dernière modification des fichiers .pt.
audit_runs_info.sort(key=lambda element: element[0], reverse=True)

if not audit_runs_info:
    print("Aucun best.pt ou last.pt trouvé dans :", audit_runs_dir)

for numero, (_, dossier, best, last) in enumerate(
    audit_runs_info[:5], start=1
):
    repere = " ← AVANT-DERNIER" if numero == 2 else ""

    print(f"\n{numero}. {dossier.name}{repere}")

    for fichier in (best, last):
        if fichier.is_file():
            date = datetime.fromtimestamp(
                fichier.stat().st_mtime,
                tz=ZoneInfo("Europe/Paris"),
            )
            print(
                f"   {fichier.name} : PRÉSENT"
                f" — {date:%d/%m/%Y à %H:%M:%S}\n"
                f"      {fichier}"
            )
        else:
            print(f"   {fichier.name} : ABSENT")

In [ ]:
from pathlib import Path

# ===== MODÈLE À TESTER =====
NOM_RUN = "nom_experiement"
FICHIER_POIDS = "best.pt"  # Ou "best.pt"
# ==========================

test_run_dir = Path("runs_thermiques").resolve() / NOM_RUN
test_weights = test_run_dir / "weights" / FICHIER_POIDS
test_output_dir = test_run_dir / f"test_final_{test_weights.stem}"

if not test_weights.is_file():
    raise FileNotFoundError(
        f"Modèle introuvable : {test_weights}"
    )

print("Modèle testé :", test_weights)
print("Résultats enregistrés dans :", test_output_dir)

test_metrics = YOLO(str(test_weights)).val(
    data=str(DATA_YAML),
    split="test",
    imgsz=EVAL_IMGSZ,
    device=PROFILES[PROFILE]["device"],
    batch=PROFILES[PROFILE]["batch"],
    workers=0,
    conf=0.001,
    iou=0.6,
    rect=False,
    max_det=MAX_DET,
    plots=True,
    project=str(test_output_dir.parent),
    name=test_output_dir.name,
    exist_ok=True,
)

display(
    pd.DataFrame([metrics_row(test_metrics)]).round(4)
)

# Afficher les graphiques disponibles.
test_plot_paths = [
    test_run_dir / "results.png",
    test_output_dir / "BoxPR_curve.png",
    test_output_dir / "confusion_matrix.png",
]

for test_plot_path in test_plot_paths:
    if test_plot_path.is_file():
        with Image.open(test_plot_path) as im:
            fig, ax = plt.subplots(figsize=(11, 7))
            ax.imshow(im)
            ax.set_title(test_plot_path.name)
            ax.axis("off")
            fig.tight_layout()
            plt.show()
            plt.close(fig)

In [ ]:
manifest_verif = pd.read_csv(
    Path(DATA_YAML).parent / "manifest.csv"
)

print("Répartition des images :")
print(manifest_verif["split"].value_counts())

print("\nNombre de dossiers par partition :")
print(manifest_verif.groupby("split")["group"].nunique())

# Un groupe doit appartenir à une seule partition.
partitions_par_groupe = (
    manifest_verif.groupby("group")["split"].nunique()
)

print(
    "\nDossiers présents dans plusieurs partitions :",
    int((partitions_par_groupe > 1).sum()),
)

print("\nPartition de chaque dossier :")
print(
    manifest_verif[["group", "split"]]
    .drop_duplicates()
    .sort_values(["split", "group"])
    .to_string(index=False)
)

In [ ]:
from PIL import Image

with Image.open("DJI_20240528140724_0010_T.JPG") as im:
    print("Format détecté :", im.format)
    print("Nombre d'images internes :", getattr(im, "n_frames", 1))

In [ ]:
from PIL import Image
from IPython.display import display

with Image.open("DJI_20240528140724_0010_T.JPG") as im:
    for i in range(im.n_frames):
        im.seek(i)
        print(f"Vue {i} — dimensions : {im.size}")
        display(im.convert("RGB"))

In [ ]:
%matplotlib inline
from pathlib import Path
from functools import lru_cache

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO


@lru_cache(maxsize=2)
def _load_detector(weights, device):
    # device fait partie de la clé du cache :
    # CPU et GPU disposent ainsi de prédicteurs distincts.
    return YOLO(weights)


def inferer_image(
    image_path,
    weights,
    *,
    frame_index=0,
    device="cpu",           # "cpu" ou 0 pour le premier GPU NVIDIA, imgz : Ou (512, 640) : ordre hauteur, largeur
    imgsz=640,              
    conf=0.05,
    iou=0.6,                # IoU de suppression des boîtes redondantes (NMS)
    max_det=300,
    show=True,
):
    image_path = Path(image_path).expanduser().resolve()
    weights = Path(weights).expanduser().resolve()

    if not image_path.is_file():
        raise FileNotFoundError(image_path)
    if not weights.is_file():
        raise FileNotFoundError(weights)
    if not (0 <= conf <= 1 and 0 <= iou <= 1):
        raise ValueError("conf et iou doivent être compris entre 0 et 1.")

    # Même préparation qu'à l'entraînement :
    # conversion en gris, puis trois canaux identiques.
    with Image.open(image_path) as im:
        n_frames = getattr(im, "n_frames", 1)

        if not 0 <= frame_index < n_frames:
            raise ValueError(
                f"frame_index={frame_index} invalide : "
                f"le fichier contient {n_frames} vue(s)."
            )

        im.seek(frame_index)

        if im.mode not in ("L", "RGB"):
            raise ValueError(
                f"Mode {im.mode} : image 8 bits en mode L ou RGB attendue."
            )

        # Conversion en gris, puis trois canaux identiques.
        rgb = im.convert("L").convert("RGB")

    detector = _load_detector(str(weights), str(device))

    result = detector.predict(
        source=rgb,
        device=device,
        imgsz=imgsz,
        rect=False,         # Canevas demandé, proportions conservées avec bordures
        conf=conf,
        iou=iou,
        classes=[1],
        max_det=max_det,
        augment=False,
        verbose=False,
    )[0]

    if show:
        result.names[1] = "p"
        # result.plot() renvoie une image BGR ; matplotlib attend du RGB.
        annotated = result.plot(labels=True, conf=True)

        plt.figure(figsize=(12, 9))
        plt.imshow(annotated[..., ::-1])
        plt.title(f"{image_path.name} — {len(result.boxes)} détection(s)")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

    return result


WEIGHTS = "runs_thermiques/nom_experiement/weights/best.pt"

images_a_tester = [
    "DJI_20240528140724_0010_T.JPG",
    "DJI_20240528142601_0025_T.JPG",
    "000001.jpg",
    "000002.jpg",
    "DJI_20240528142843_0079_T.JPG",
    "DJI_20240528143023_0112_T.JPG",
    "DJI_20240612213511_0018_T.JPG",
    "DJI_20240722175454_0434_T.JPG",
    "DJI_20240722175540_0457_T.JPG",
    "DJI_20240722175858_0556_T.JPG",
    "DJI_20240722175926_0570_T.JPG",
]

for numero, image_path in enumerate(images_a_tester, start=1):
    print(f"\n[{numero}/{len(images_a_tester)}] {image_path}")

    try:
        result = inferer_image(
            image_path=image_path,
            weights=WEIGHTS,
            frame_index=0,
            device="cpu",
            imgsz=640,
            conf=0.1,
            iou=0.6,
            show=False,  # Affichage géré ci-dessous.
        )
    except (OSError, ValueError) as exc:
        print(f"Image non traitée : {exc}")
        continue

    print("Boîtes [x1, y1, x2, y2] :")
    print(result.boxes.xyxy.cpu().numpy())

    print("Scores :")
    print(result.boxes.conf.cpu().numpy())

    result.names[1] = "p"
    annotated = result.plot(labels=True, conf=True)

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(annotated[..., ::-1])
    ax.set_title(
        f"{numero}/{len(images_a_tester)} — "
        f"{Path(image_path).name} — "
        f"{len(result.boxes)} détection(s)"
    )
    ax.axis("off")
    fig.tight_layout()

    plt.show()
    plt.close(fig)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from PIL import Image


# ===== CHEMINS À MODIFIER SI NÉCESSAIRE =====
SUPP_IMAGES = Path("image_test_supp").expanduser().resolve()
SUPP_LABELS = Path("labels_test_supp").expanduser().resolve()

# Nouveau dossier réservé à ce jeu de test.
SUPP_PREPARED = Path("dataset_test_supp_prepare").resolve()

# 0 fonctionne aussi pour les images avec une seule vue interne.
# Les annotations doivent correspondre à la vue sélectionnée.
SUPP_FRAME_INDEX = 0
# ===========================================
def convertir_labels_test(text, label_path):
    """Convertit la classe source 0 en classe 1 et vérifie les coordonnées."""
    lignes_converties = []

    for numero, ligne in enumerate(text.splitlines(), start=1):
        if not ligne.strip():
            continue

        parts = ligne.split()

        if len(parts) != 5:
            raise ValueError(
                f"{label_path}, ligne {numero} : 5 valeurs attendues."
            )

        try:
            valeurs = np.array(parts, dtype=float)
        except ValueError as exc:
            raise ValueError(
                f"{label_path}, ligne {numero} : valeurs non numériques."
            ) from exc

        if not np.isfinite(valeurs).all():
            raise ValueError(
                f"{label_path}, ligne {numero} : valeur non finie."
            )

        classe, cx, cy, largeur, hauteur = valeurs

        if classe != 0:
            raise ValueError(
                f"{label_path}, ligne {numero} : classe source 0 attendue."
            )

        if not (
            0 <= cx <= 1
            and 0 <= cy <= 1
            and 0 < largeur <= 1
            and 0 < hauteur <= 1
        ):
            raise ValueError(
                f"{label_path}, ligne {numero} : coordonnées invalides."
            )

        # Seule la classe change ; les coordonnées restent identiques.
        lignes_converties.append("1 " + " ".join(parts[1:]))

    return "\n".join(lignes_converties), len(lignes_converties)

def preparer_test_supp(images_dir, labels_dir, target, frame_index=0):
    images_dir = Path(images_dir).resolve()
    labels_dir = Path(labels_dir).resolve()
    target = Path(target).resolve()

    if not images_dir.is_dir():
        raise FileNotFoundError(f"Dossier images absent : {images_dir}")

    if not labels_dir.is_dir():
        raise FileNotFoundError(f"Dossier labels absent : {labels_dir}")

    if target.exists():
        raise FileExistsError(
            f"{target} existe déjà. Pour le réutiliser, "
            "exécuter uniquement la cellule d'évaluation. "
            "Pour une nouvelle préparation, choisir un autre nom."
        )

    extensions = {
        ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".mpo"
    }

    images = sorted(
        p for p in images_dir.iterdir()
        if p.is_file() and p.suffix.lower() in extensions
    )

    if not images:
        raise ValueError(f"Aucune image trouvée dans {images_dir}")

    # Vérifier les annotations avant de créer le dataset.
    records = []
    textes = []

    for numero, image_path in enumerate(images):
        label_path = labels_dir / f"{image_path.stem}.txt"

        if label_path.exists() and not label_path.is_file():
            raise ValueError(
                f"Le chemin d'annotation n'est pas un fichier : {label_path}"
            )

        if not label_path.is_file():
            statut = "absent"
            text = ""
            n_boxes = 0
        else:
            text = label_path.read_text(encoding="utf-8-sig")

            if not text.strip():
                statut = "vide"
                text = ""
                n_boxes = 0
            else:
                statut = "annote"
                text, n_boxes = convertir_labels_test(text, label_path)

        prepared_name = f"{numero:08d}"

        records.append({
            "source_image": str(image_path),
            "source_label": str(label_path),
            "statut_annotation": statut,
            "n_boxes": n_boxes,
            "prepared_image": str(
                target / "images" / "test" / f"{prepared_name}.png"
            ),
            "prepared_label": str(
                target / "labels" / "test" / f"{prepared_name}.txt"
            ),
            "frame_index": frame_index,
        })
        textes.append(text)

    manifest = pd.DataFrame(records)

    print("Images trouvées :", len(manifest))
    print(
        "Images sans fichier label associé :",
        int((manifest["statut_annotation"] == "absent").sum()),
    )
    print(
        "Images avec fichier label associé vide :",
        int((manifest["statut_annotation"] == "vide").sum()),
    )
    print(
        "Images avec annotations non vides :",
        int((manifest["statut_annotation"] == "annote").sum()),
    )
    print("Personnes annotées :", int(manifest["n_boxes"].sum()))

    (target / "images" / "test").mkdir(parents=True)
    (target / "labels" / "test").mkdir(parents=True)

    for record, text in zip(records, textes):
        image_path = Path(record["source_image"])

        with Image.open(image_path) as im:
            n_frames = getattr(im, "n_frames", 1)

            if not 0 <= frame_index < n_frames:
                raise ValueError(
                    f"{image_path} : vue {frame_index} invalide, "
                    f"{n_frames} vue(s) disponible(s)."
                )

            im.seek(frame_index)

            if im.getexif().get(274, 1) != 1:
                raise ValueError(
                    f"{image_path} : corriger l'orientation EXIF "
                    "et vérifier les annotations avant la préparation."
                )

            if im.mode not in ("L", "RGB"):
                raise ValueError(
                    f"{image_path} : mode {im.mode} non accepté. "
                    "Image 8 bits L ou RGB attendue."
                )

            if min(im.size) < 10:
                raise ValueError(f"Image trop petite : {image_path}")

            im.convert("L").convert("RGB").save(
                record["prepared_image"]
            )

        # Classe 1 conservée ; fichier vide pour les négatives.
        Path(record["prepared_label"]).write_text(
            text.rstrip() + "\n" if text.strip() else "",
            encoding="utf-8",
        )

    config = {
        "path": str(target),
        # Clés requises pour la compatibilité du format YAML.
        # Ce dataset est exclusivement destiné à l'évaluation.
        "train": None,
        "val": "images/test",
        "test": "images/test",
        "names": {0: "inutilisee", 1: "personne"},
        "channels": 3,
    }

    data_yaml = target / "data.yaml"
    data_yaml.write_text(
        yaml.safe_dump(config, sort_keys=False),
        encoding="utf-8",
    )

    # Permet de retrouver les noms originaux et les labels absents/vides.
    manifest.to_csv(target / "manifest.csv", index=False)

    print("\nTotal d'images préparées :", len(manifest))
    print("Dataset créé :", target)
    print("Configuration :", data_yaml)

    return data_yaml


SUPP_DATA_YAML = preparer_test_supp(
    SUPP_IMAGES,
    SUPP_LABELS,
    SUPP_PREPARED,
    frame_index=SUPP_FRAME_INDEX,
)

In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image
from ultralytics import YOLO


# ===== PARAMÈTRES À MODIFIER =====
SUPP_NOM_RUN = "nom_experiment"
SUPP_FICHIER_POIDS = "best.pt"  # Ou "best.pt"

SUPP_DATA_YAML = (
    Path("dataset_test_supp_prepare").resolve() / "data.yaml"
)

SUPP_IMGSZ = 640
SUPP_BATCH = 4
SUPP_MAX_DET = 300
# =================================
SUPP_IMGSZ = 1664

supp_run_dir = Path("runs_thermiques").resolve() / SUPP_NOM_RUN
supp_weights = supp_run_dir / "weights" / SUPP_FICHIER_POIDS

if not supp_weights.is_file():
    raise FileNotFoundError(f"Modèle introuvable : {supp_weights}")

if not SUPP_DATA_YAML.is_file():
    raise FileNotFoundError(
        f"Dataset préparé introuvable : {SUPP_DATA_YAML}"
    )

print("Modèle testé :", supp_weights)
print("Dataset utilisé :", SUPP_DATA_YAML)

supp_detector = YOLO(str(supp_weights))

supp_metrics = supp_detector.val(
    data=str(SUPP_DATA_YAML),
    split="test",
    device="cpu",
    imgsz=SUPP_IMGSZ,
    batch=SUPP_BATCH,
    workers=0,
    fraction=1.0,
    classes=[1],
    conf=0.001,        # Seuil bas pour calculer les courbes et la mAP
    iou=0.6,           # Suppression des boîtes redondantes
    rect=False,
    max_det=SUPP_MAX_DET,
    plots=True,
    project=str(supp_run_dir),
    name=f"test_supp_{supp_weights.stem}",
    exist_ok=False,    # Nouvelle sortie pour chaque évaluation
)

supp_scores = pd.DataFrame([{
    "precision": float(supp_metrics.box.mp),
    "recall": float(supp_metrics.box.mr),
    "map50": float(supp_metrics.box.map50),
    "map50_95": float(supp_metrics.box.map),
}])

display(supp_scores.round(4))

# Utilise le dossier réellement créé, y compris son éventuel suffixe.
supp_output_dir = Path(supp_metrics.save_dir)
supp_scores.to_csv(supp_output_dir / "metriques.csv", index=False)

print("Résultats enregistrés dans :", supp_output_dir)

for supp_plot_name in (
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "confusion_matrix.png",
):
    supp_plot_path = supp_output_dir / supp_plot_name

    if supp_plot_path.is_file():
        with Image.open(supp_plot_path) as im:
            fig, ax = plt.subplots(figsize=(11, 7))
            ax.imshow(im)
            ax.set_title(supp_plot_name)
            ax.axis("off")
            fig.tight_layout()
            plt.show()
            plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from matplotlib.patches import Rectangle

def afficher_resultats_test_supp(
    repartition=(5, 5, 5, 5),
    confiance=0.1,
):
    manifest = pd.read_csv(
        SUPP_DATA_YAML.parent / "manifest.csv"
    ).reset_index(drop=True)

    if manifest.empty:
        raise ValueError("Le manifeste ne contient aucune image.")

    if len(repartition) != 4 or any(n < 0 for n in repartition):
        raise ValueError("Indiquer quatre nombres positifs ou nuls.")

    # Diviser les positions en quatre parties consécutives.
    quarts = np.array_split(np.arange(len(manifest)), 4)
    morceaux_selectionnes = []

    for numero_quart, (indices, nombre) in enumerate(
        zip(quarts, repartition), start=1
    ):
        quart = manifest.iloc[indices]
        nombre_retenu = min(nombre, len(quart))

        extrait = quart.sample(
            n=nombre_retenu,
            random_state=42 + numero_quart,
        ).copy()

        extrait["quart"] = numero_quart
        morceaux_selectionnes.append(extrait)

        print(
            f"Quart {numero_quart} : "
            f"{len(quart)} images disponibles, "
            f"{nombre_retenu} sélectionnées"
        )

    selection = pd.concat(
        morceaux_selectionnes,
        ignore_index=True,
    )

    if selection.empty:
        print("Aucune image sélectionnée.")
        return

    print(f"\nTotal : {len(selection)} images sélectionnées")

    # Afficher dix images par figure.
    for debut in range(0, len(selection), 10):
        groupe = selection.iloc[debut:debut + 10]
        n_lignes = (len(groupe) + 1) // 2

        fig, axes = plt.subplots(
            nrows=n_lignes,
            ncols=2,
            figsize=(16, 4.5 * n_lignes),
        )
        axes = np.asarray(axes).reshape(-1)

        for ax, (numero, ligne) in zip(axes, groupe.iterrows()):
            with Image.open(ligne["prepared_image"]) as im:
                image_rgb = im.convert("RGB")

            prediction = supp_detector.predict(
                source=image_rgb,
                device="cpu",
                imgsz=SUPP_IMGSZ,
                rect=False,
                conf=confiance,
                iou=0.6,
                classes=[1],
                max_det=SUPP_MAX_DET,
                augment=False,
                verbose=False,
            )[0]

            prediction.names = dict(prediction.names)
            prediction.names[1] = "p"

            image_annotee = prediction.plot(
                labels=True,
                conf=True,
                line_width=1,
                font_size=10,
            )

            ax.imshow(image_annotee[..., ::-1])

            # Superposer les annotations réelles en vert pointillé.
            largeur_image, hauteur_image = image_rgb.size
            label_path = Path(ligne["prepared_label"])

            if not label_path.is_file():
                raise FileNotFoundError(f"Annotations introuvables : {label_path}")

            for texte in label_path.read_text(encoding="utf-8-sig").splitlines():
                if not texte.strip():
                    continue

                classe, cx, cy, largeur, hauteur = map(float, texte.split())

                # Les labels préparés utilisent la classe 1 pour les personnes.
                if classe != 1:
                    continue

                # Coordonnées YOLO normalisées → coordonnées en pixels.
                x = (cx - largeur / 2) * largeur_image
                y = (cy - hauteur / 2) * hauteur_image
                w = largeur * largeur_image
                h = hauteur * hauteur_image

                ax.add_patch(
                    Rectangle(
                        (x, y), w, h,
                        fill=False,
                        edgecolor="lime",
                        linewidth=1.5,
                        linestyle="--",
                    )
                )

                ax.text(
                    x, max(0, y - 3),
                    "Annotation",
                    color="lime",
                    fontsize=8,
                    va="bottom",
                    bbox=dict(facecolor="black", alpha=0.6, edgecolor="none"),
                )
            ax.set_title(
                f"{numero + 1}/{len(selection)} — "
                f"Quart {int(ligne['quart'])} — "
                f"{Path(ligne['source_image']).name}\n"
                f"{len(prediction.boxes)} détection(s) — "
                f"{int(ligne['n_boxes'])} personne(s) annotée(s)",
                fontsize=10,
            )
            ax.axis("off")

        for ax in axes[len(groupe):]:
            ax.axis("off")

        fig.tight_layout()
        plt.show()
        plt.close(fig)


# Nombre d'images tirées au hasard dans chaque quart :
#                       1er  2e  3e  4e
afficher_resultats_test_supp(
    repartition=(5, 5, 5, 5),
    confiance=0.1,
)